# Spark Logistic Regression Model Training Script

This notebook provides a clean implementation for loading features from a table and training a Logistic Regression model using Apache Spark MLlib.

## Overview
- Load data from Spark table/DataFrame
- Prepare features for machine learning
- Train Logistic Regression model
- Evaluate model performance
- Save trained model

In [1]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from clickhouse_driver import Client
import configparser
from pathlib import Path

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]
CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,  
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}
url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user'] 
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

try:
    spark.stop()
    print("� Stopped existing Spark session")
except:
    pass

spark = SparkSession.builder \
    .appName("data_loading") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "100g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()


25/10/31 18:05:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## 1. Load Features from ClickHouse

Load the fraud detection features from the ClickHouse `stixor_fraud_features_distributed` table using the context from the fraud dataset profiling notebook.

In [4]:
parquet_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_accounts_with_types"
fraud_accounts_df = spark.read.parquet(parquet_path)
fraud_accounts_df.show(5)

25/10/31 17:42:24 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


+--------------------+--------------------+
|       a_c_reference|   account_type_name|
+--------------------+--------------------+
|A4IodXocoJOsy7gXH...|    Customer Account|
|eP+92RAjQ+bwD8u8I...|    Customer Account|
|/HbE5CTocvF9+aJxA...|    Customer Account|
|v3jU6QuWv8diZu3lc...|    Customer Account|
|fPgIKI/bJEDGPQ8eO...|Payment Gateway A...|
+--------------------+--------------------+
only showing top 5 rows


In [ ]:
customer_fraud_accounts_df = fraud_accounts_df.filter(fraud_accounts_df.account_type_name == 'Customer Account')

In [8]:
fraud_accounts_df.count()

5519

In [ ]:
customer_fraud_accounts_df.count()

5276

In [9]:
from clickhouse_driver import Client

# Create a table in ClickHouse and insert the customer_fraud_accounts_df DataFrame

# 1. Define ClickHouse table schema based on DataFrame schema
table_name = "fraud_customer_accounts"
columns = ", ".join([f"{field.name} String" for field in customer_fraud_accounts_df.schema.fields])

# 2. Connect to ClickHouse and create the table if not exists

client = Client(
    host=CLICKHOUSE_CONFIG['host'],
    port=CLICKHOUSE_CONFIG['port'],
    user=CLICKHOUSE_CONFIG['user'],
    password=CLICKHOUSE_CONFIG['password'],
    database=CLICKHOUSE_CONFIG['database']
)

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    {columns}
) ENGINE = MergeTree()
ORDER BY tuple()
"""

client.execute(create_table_sql)
print(f"✅ Table '{table_name}' created or already exists in ClickHouse.")

# 3. Collect data from Spark DataFrame and insert into ClickHouse
data = customer_fraud_accounts_df.toPandas()
records = data.to_records(index=False)
rows = [tuple(row) for row in records]

if rows:
    insert_sql = f"INSERT INTO {table_name} VALUES"
    client.execute(insert_sql, rows)
    print(f"✅ Inserted {len(rows)} rows into '{table_name}' in ClickHouse.")
else:
    print("⚠️ No data to insert.")

✅ Table 'fraud_customer_accounts' created or already exists in ClickHouse.
✅ Inserted 5276 rows into 'fraud_customer_accounts' in ClickHouse.


In [23]:
query = f"""
    SELECT count(*)
    FROM public.stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
      AND (ac_from IN (select a_c_reference from fraud_customer_accounts)
           OR ac_to IN (select a_c_reference from fraud_customer_accounts)
"""

print(query)


    SELECT count(*)
    FROM public.stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '2025-01-01' AND '2025-07-31'
      AND (ac_from IN (select a_c_reference from fraud_customer_accounts)
           OR ac_to IN (select a_c_reference from fraud_customer_accounts)



In [ ]:
# Use the fraud_accounts_df DataFrame (contains a_c_reference) to filter the features table
# fraud_ac_refs = [row.a_c_reference for row in fraud_accounts_df.select("a_c_reference").collect()]
start_date='2025-01-01'
end_date='2025-07-31'
num_partitions = 210


query = f"""
SELECT *
FROM public.stixor_fraud_features_distributed
WHERE (((cutoff_date >= '2025-01-01') AND (cutoff_date <= '2025-07-31')) AND (ac_from GLOBAL IN (
    SELECT a_c_reference
    FROM public.fraud_distributed
    GLOBAL INNER JOIN public.stixor_mbar_v_distributed b
    ON fraud.fraud_msisdn = b.a_c_reference
    WHERE b.account_type_name = 'Customer Account'
))) OR (ac_to GLOBAL IN (
    SELECT distinct a_c_reference
    FROM public.fraud_distributed as fraud
    GLOBAL INNER JOIN public.stixor_mbar_v_distributed b
    ON fraud.fraud_msisdn = b.a_c_reference
    WHERE b.account_type_name = 'Customer Account'
))
"""
    #   AND (ac_from IN ({','.join([f"'{ref}'" for ref in fraud_ac_refs])})
    #        OR ac_to IN ({','.join([f"'{ref}'" for ref in fraud_ac_refs])}))

subquery = f"""
(
    {query}
) AS fraud_data
"""

try:
    df = (spark.read
        .format('jdbc')
        .option('driver', driver)
        .option('url', url)
        .option('user', user)
        .option('password', password)
        .option('dbtable', subquery)
        .option('fetchsize', '100000')
        .option("partitionColumn", "cutoff_date")
        .option('lowerBound', start_date)
        .option('upperBound', end_date)
        .option('numPartitions', str(num_partitions))
        .load())
    
    print(f"\n📦 Caching DataFrame in memory...")
    df.cache()
    print(f"⏳ Counting rows (this triggers data loading)...")
    total_rows = df.count()
    num_partitions = df.rdd.getNumPartitions()
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    print(f"\n✅ DATA LOADED SUCCESSFULLY!")
except Exception as e:
    print(f"\n❌ ERROR LOADING DATA!")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()
    raise



📦 Caching DataFrame in memory...


25/10/31 18:05:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


⏳ Counting rows (this triggers data loading)...



❌ ERROR LOADING DATA!
Error: name 'datetime' is not defined


Traceback (most recent call last):                                              
  File "/tmp/ipykernel_3451540/4045516432.py", line 48, in <module>
    end_time = datetime.now()
NameError: name 'datetime' is not defined


NameError: name 'datetime' is not defined

In [5]:
df.show(10)

+-----------+--------------------+--------------------+----------+-------------------+-----------+----------+-----------+--------------------+-------------+-----------+-------+--------------------+-----------+---------+-----------------------+-------------------------+---------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+----------------------------+-----------------+-------------------+--------------------+----------------------+----------------+----------+---------+--------------+------------+----------------+-----------+-----------+----------+--------+-----------------+---------------+-------------------+------------------+--------------+------------------+-----------+-------------------+------------------+-----------------+-----------------+------------------------+----------------------+-------------------+-----------------------+------------------------+----------------------------

In [ ]:
import os

# Calculate the size of the DataFrame in GB and save it as a parquet file

# Save DataFrame to parquet
parquet_output_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot"
df.write.mode("overwrite").parquet(parquet_output_path)

# Calculate size in GB
def get_dir_size_gb(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 ** 3)

df_size_gb = get_dir_size_gb(parquet_output_path)
print(f"DataFrame size on disk: {df_size_gb:.2f} GB")b

DataFrame size on disk: 0.02 GB


In [8]:
# Save DataFrame to CSV
csv_output_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_csv"
df.coalesce(1).write.mode("overwrite").option("header", True).csv(csv_output_path)
print(f"DataFrame saved as CSV to: {csv_output_path}")


DataFrame saved as CSV to: /root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_csv


In [9]:
import pandas as pd
import zipfile

# Convert Spark DataFrame to Pandas DataFrame
df_pd = df.toPandas()

# Write to CSV
csv_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_full.csv"
df_pd.to_csv(csv_path, index=False)

# Zip the CSV file
zip_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_full.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(csv_path, arcname="df_snapshot_full.csv")

print(f"✅ DataFrame written to CSV: {csv_path}")
print(f"✅ CSV file zipped at: {zip_path}")

✅ DataFrame written to CSV: /root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_full.csv
✅ CSV file zipped at: /root/research-dir/dev/jazzcash-fraud-detection/data/df_snapshot_full.zip


In [14]:
print(query)


    SELECT count(*)
    FROM public.stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '2025-01-01' AND '2025-07-31'
      AND (ac_from IN ('A4IodXocoJOsy7gXHlR2yA==','eP+92RAjQ+bwD8u8I2w4FA==','/HbE5CTocvF9+aJxApc+aQ==','v3jU6QuWv8diZu3lc2I4GA==','fPgIKI/bJEDGPQ8eOtPT9A==','VzfjB//rB1H/jwmEMApF0g==','yuY+biic0NvqJFT7npI4DQ==','AB4q49vtuyW9MqsKMHGxMw==','lMfBiKFFs7OhYhl1wVC+DA==','6b59V+uYErkLcqa59AGXHQ==','Zddn1Wc0xQx9S+BHhieDnw==','B8bU9B8H9dXaLr036nZeRw==','ZUhAG7MAkRZmeYeeyxijxg==','Y3W15qUp3I7Zdz5mUfw6xw==','1qKcjHCUijch9Rx4nBFBhg==','qSQvbHd/6TxM1AW0mW6Mxw==','1V8jN/NcASeSkd9xhcvv9w==','WfSfBY32yGJAK0sWl69itg==','j8Oq1JGEJPtXxMDoK42nXg==','KiKxPVkIyE548Hd1DuDuCA==','bH3ZpoiiObRVaZMNjkUt8Q==','9poehqOvopkN2z12Xj1rfA==','i8P7NKZ6j9Hw3pYfloiw/g==','F/PKJxSEsIRHny3IkfvU0w==','J1p5zCyPeLGM0XZFnQx7hA==','LiMiXgEbT0FrZVhzdhLzhA==','F67RyXyFFKMtnujTy4VUUw==','471jqpL+qURsTvtHa019pg==','QwLxz1E05BS3GEt0CAzJUQ==','cOVIaE+itl6E6TI2ivbQJw==','oBpw50mO+LzY8gVyFt1V+g==','2F9DBJUX

In [ ]:
import pyspark.sql.functions as F
from datetime import datetime

start_date = '2025-01-01'
end_date = '2025-07-31'
num_partitions = 30

selected_cols = [
'cutoff_date',
'fraud_flag',
 'trx_channel',
 'trx_type',
 'start_balance',
 'trx_amt',
 'mbar_registered_channel',
 'mbar_a_c_status',
 'mbar_a_c_level',
 'mbar_account_type_name',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'is_night',
 'is_business_hours',
 'is_unusual_hour',
 'night_weekend_combo',
 'start_balance_log',
 'txn_txns_3d',
 'txn_total_amount_3d',
 'txn_avg_amount_3d',
 'txn_max_amount_3d',
 'txn_min_amount_3d',
 'txn_unique_recipients_3d',
 'txn_unique_channels_3d',
 'txn_unique_types_3d',
 'txn_is_high_activity_3d',
 'txn_multi_channel_recent',
 'txn_amount_deviation_from_avg',
 'txn_night_txns_3d',
 'txn_weekend_txns_3d',
 'channel_new_jc_app',
 'channel_ussd',
 'channel_ussd_api',
 'channel_payment_gateway',
 'channel_mobile_app',
 'type_transfer_c2c',
 'type_transfer_c2b',
 'type_bill_payment',
 'type_mobile_load',
 'user_total_txns_3d',
 'user_total_amount_3d',
 'user_avg_amount_3d',
 'user_median_amount_3d',
 'user_max_amount_3d',
 'user_min_amount_3d',
 'user_unique_recipients_3d',
 'user_unique_channels_3d',
 'user_unique_types_3d',
 'user_total_txns_7d',
 'user_total_amount_7d',
 'user_avg_amount_7d',
 'user_median_amount_7d',
 'user_max_amount_7d',
 'user_min_amount_7d',
 'user_unique_recipients_7d',
 'user_unique_channels_7d',
 'user_unique_types_7d',
 'user_most_used_channel_7d',
 'user_last_used_channel',
 'user_channel_diversity_score_7d',
 'user_most_used_type_7d',
 'user_last_used_type',
 'user_type_diversity_score_7d',
 'user_night_txns_7d',
 'user_weekend_txns_7d',
 'user_peak_hour_txns_7d',
 'user_off_peak_hour_txns_7d',
 'user_avg_start_balance_7d',
 'user_avg_end_balance_7d',
 'user_min_balance_7d',
 'user_max_balance_7d',
 'user_balance_volatility_7d',
 'user_avg_amount_per_recipient_7d',
 'user_max_amount_to_single_recipient_7d',
 'user_recipient_concentration_ratio_7d',
 'user_avg_time_between_txns_7d',
 'user_txn_frequency_score_7d',
 'user_days_since_last_txn']


query = f"""
    SELECT {', '.join(selected_cols)}
    FROM stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
        AND mbar_account_type_name = 'Customer Account'
"""

subquery = f"""
(
    {query}
) AS fraud_data
"""

start_time = datetime.now()
try:
    df = (spark.read
        .format('jdbc')
        .option('driver', driver)
        .option('url', url)
        .option('user', user)
        .option('password', password)
        .option('dbtable', subquery)
        .option('fetchsize', '100000')  # Fetch 100k rows at a time
        .option("partitionColumn", "cutoff_date") \
        .option('lowerBound', start_date)  # Lower bound of partition column
        .option('upperBound', end_date)    # Upper bound of partition column
        .option('numPartitions', str(num_partitions))  # Number of partitions
        .load())
    
    # Cache the DataFrame for better performance
    print(f"\n📦 Caching DataFrame in memory...")
    df.cache()
    
    # Trigger action to load data
    print(f"⏳ Counting rows (this triggers data loading)...")
    total_rows = df.count()
    num_partitions = df.rdd.getNumPartitions()
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print(f"\n✅ DATA LOADED SUCCESSFULLY!")
    
except Exception as e:
    print(f"\n❌ ERROR LOADING DATA!")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()
    raise


25/10/31 10:28:20 WARN JDBCRelation: The number of partitions is reduced because the specified number of partitions is less than the difference between upper bound and lower bound. Updated number of partitions: 29; Input number of partitions: 30; Lower bound: '2025-06-01'; Upper bound: '2025-06-30'.



📦 Caching DataFrame in memory...


25/10/31 10:28:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


⏳ Counting rows (this triggers data loading)...



✅ DATA LOADED SUCCESSFULLY!


## 2 Feature Assembling and Scaling

In [5]:
# Convert string (categorical) features to numerical form using StringIndexer and OneHotEncoder
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Feature analysis based on fraud dataset profiling insights
print("� Analyzing feature characteristics...")

TARGET_COLUMN='fraud_flag'
# Check target variable distribution
print(f"\n📈 Target Variable ({TARGET_COLUMN}) Distribution:")
df.groupBy(TARGET_COLUMN).count().show()

# Get feature columns (exclude non-predictive columns from fraud analysis)
excluded_columns = [
    TARGET_COLUMN,  # Target variable
    'processed_date',  # Time identifier
    'ac_from',  # Account identifiers
    'ac_to', 
    'msisdn_from',
    'msisdn_to',
    'transaction_uuid'  # Transaction identifiers
]

# Get all numerical feature columns based on fraud profiling analysis
all_feature_cols = [col_name for col_name in df.columns if col_name not in excluded_columns]

# Identify string (categorical) columns in all_feature_cols
string_cols = []
for field in df.schema.fields:
    if field.name in all_feature_cols and field.dataType.typeName() == 'string':
        string_cols.append(field.name)

print(f"\n🔤 String (categorical) features to encode: {string_cols}")

# Replace empty strings in string columns with 'UNKNOWN'
from pyspark.sql.functions import when
for col_name in string_cols:
    df = df.withColumn(col_name, when((F.col(col_name) == "") | F.col(col_name).isNull(), "UNKNOWN").otherwise(F.col(col_name)))

# Index and encode string columns
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in string_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe", handleInvalid="keep") for col in string_cols
]

# Apply indexers and encoders sequentially
from pyspark.ml import Pipeline

cat_pipeline = Pipeline(stages=indexers + encoders)
df_cat = cat_pipeline.fit(df).transform(df)

# Replace original string columns in all_feature_cols with their OHE columns
modeling_features = [
    f"{col}_ohe" if col in string_cols else col for col in all_feature_cols
]

# Remove any columns that are not present in df_cat (e.g., if OHE dropped some columns)
modeling_features = [col for col in modeling_features if col in df_cat.columns]

print(f"\n🧮 Final modeling features (after encoding): {modeling_features}")

# Use df_cat as the cleaned DataFrame for further processing
df_clean = df_cat


🔤 String (categorical) features to encode: ['trx_channel', 'trx_type', 'mbar_registered_channel', 'mbar_a_c_status', 'mbar_a_c_level', 'mbar_account_type_name', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_most_used_type_7d', 'user_last_used_type']



🧮 Final modeling features (after encoding): ['cutoff_date', 'trx_channel_ohe', 'trx_type_ohe', 'start_balance', 'trx_amt', 'mbar_registered_channel_ohe', 'mbar_a_c_status_ohe', 'mbar_a_c_level_ohe', 'mbar_account_type_name_ohe', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'is_business_hours', 'is_unusual_hour', 'night_weekend_combo', 'start_balance_log', 'txn_txns_3d', 'txn_total_amount_3d', 'txn_avg_amount_3d', 'txn_max_amount_3d', 'txn_min_amount_3d', 'txn_unique_recipients_3d', 'txn_unique_channels_3d', 'txn_unique_types_3d', 'txn_is_high_activity_3d', 'txn_multi_channel_recent', 'txn_amount_deviation_from_avg', 'txn_night_txns_3d', 'txn_weekend_txns_3d', 'channel_new_jc_app', 'channel_ussd', 'channel_ussd_api', 'channel_payment_gateway', 'channel_mobile_app', 'type_transfer_c2c', 'type_transfer_c2b', 'type_bill_payment', 'type_mobile_load', 'user_total_txns_3d', 'user_total_amount_3d', 'user_avg_amount_3d', 'user_median_amount_3d', 'user_max_amount_3d', 'user_min_amoun

In [6]:
modeling_features = modeling_features[1:]

## 3. Feature Engineering & Preparation

Prepare features for machine learning by creating feature vectors and splitting the data.

In [7]:
# Create feature vectors using selected features
print("🔧 Creating feature vectors for Spark MLlib...")

# Create feature vector using VectorAssembler
assembler = VectorAssembler(
    inputCols=modeling_features,
    outputCol="raw_features",
    handleInvalid="skip"  # Skip rows with invalid values
)

# Apply the assembler
print(f"   • Assembling {len(modeling_features)} features into vector...")
df_assembled = assembler.transform(df_clean)
print(f"✅ Feature vector created")

# Scale features using StandardScaler (important for logistic regression)
print("📏 Scaling features...")
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,  # Scale to unit variance
    withMean=True  # Center the data
)

# Fit and transform the scaler
print("   • Fitting scaler on training data...")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)
print("✅ Features scaled successfully")

# Prepare final dataset for ML training
df_final = df_scaled.select("features", col(TARGET_COLUMN).alias("label"))

print(f"✅ Final dataset prepared:")
print(f"   • Features: Vector of {len(modeling_features)} elements")
print(f"   • Label: Binary (0=Legitimate, 1=Fraud)")
print(f"   • Records: {df_final.count():,}")

# Show sample of the prepared data
print(f"\n📋 Sample of Prepared Data:")
df_final.show(3, truncate=False)

🔧 Creating feature vectors for Spark MLlib...
   • Assembling 77 features into vector...
✅ Feature vector created
📏 Scaling features...
   • Fitting scaler on training data...


✅ Features scaled successfully
✅ Final dataset prepared:
   • Features: Vector of 77 elements
   • Label: Binary (0=Legitimate, 1=Fraud)


   • Records: 125,555,734

📋 Sample of Prepared Data:
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Split data for training and evaluation based on time to avoid data leakage
print("✂️ Splitting data into training and test sets based on time (no leakage)...")

# Define cutoff date for train/test split (adjust as needed)
cutoff_date = '2025-06-20'  # All records before or on this date go to train, after to test

# Ensure cutoff_date column is in the correct format (string or date)
# If cutoff_date is string, comparison works; if date, cast as needed

df_train = df_final.join(df_clean.select('cutoff_date'), on=df_final.rdd.zipWithIndex().map(lambda x: x[1]).collect(), how='left')

df_train = df_final.join(df_clean.select('cutoff_date'), df_final.rdd.zipWithIndex().map(lambda x: x[1]).collect(), 'left')

# Add cutoff_date column to df_final for splitting
from pyspark.sql.functions import col as spark_col

df_final_with_date = df_final.withColumn('cutoff_date', df_clean['cutoff_date'])

train_data = df_final_with_date.filter(spark_col('cutoff_date') <= cutoff_date).drop('cutoff_date')
test_data = df_final_with_date.filter(spark_col('cutoff_date') > cutoff_date).drop('cutoff_date')

# Cache datasets for performance
train_data.cache()
test_data.cache()

print(f"📊 Data Split Summary:")
train_count = train_data.count()
test_count = test_data.count()
print(f"   • Training set: {train_count:,} records ({train_count/(train_count+test_count)*100:.1f}%)")
print(f"   • Test set: {test_count:,} records ({test_count/(train_count+test_count)*100:.1f}%)")

# Check class distribution in both sets
print(f"\n📈 Class Distribution:")

print("Training Set:")
train_dist = train_data.groupBy("label").count().collect()
for row in train_dist:
    label = "Fraud" if row['label'] == 1 else "Legitimate" 
    count = row['count']
    percentage = (count / train_count) * 100
    print(f"   • {label}: {count:,} ({percentage:.1f}%)")

print("Test Set:")
test_dist = test_data.groupBy("label").count().collect()
for row in test_dist:
    label = "Fraud" if row['label'] == 1 else "Legitimate"
    count = row['count'] 
    percentage = (count / test_count) * 100
    print(f"   • {label}: {count:,} ({percentage:.1f}%)")

print("✅ Data split completed and cached for optimal performance")

✂️ Splitting data into training and test sets based on time (no leakage)...


## 4. Logistic Regression Model Training

Train the Logistic Regression model using Spark MLlib.

In [ ]:
# Configure Logistic Regression model optimized for fraud detection
print("🤖 Configuring Logistic Regression for fraud detection...")

# Calculate class weights for imbalanced dataset (from fraud profiling insights)
fraud_count = train_data.filter(col("label") == 1).count()
legitimate_count = train_data.filter(col("label") == 0).count()
class_ratio = legitimate_count / fraud_count

print(f"📊 Class Imbalance Analysis:")
print(f"   • Legitimate transactions: {legitimate_count:,}")
print(f"   • Fraud transactions: {fraud_count:,}")
print(f"   • Class ratio (Legit/Fraud): {class_ratio:.2f}:1")

# Configure Logistic Regression with optimal parameters for fraud detection
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label", 
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,  # Sufficient iterations for convergence
    regParam=0.01,  # L2 regularization to prevent overfitting
    elasticNetParam=0.0,  # Pure L2 regularization (Ridge)
    threshold=0.5,  # Default threshold (will optimize later)
    standardization=False,  # Features already standardized
    aggregationDepth=2,  # For better performance on large datasets
    family="binomial"  # Binary classification
)

print(f"✅ Logistic Regression configured with fraud detection optimizations:")
print(f"   • Algorithm: Binomial Logistic Regression")
print(f"   • Max Iterations: {lr.getMaxIter()}")
print(f"   • Regularization (L2): {lr.getRegParam()}")
print(f"   • Decision Threshold: {lr.getThreshold()}")
print(f"   • Feature Standardization: Disabled (pre-scaled)")
print(f"   • Optimization: LBFGS (default, good for medium datasets)")

print(f"\n🎯 Ready for training on {len(modeling_features)} engineered features from fraud profiling analysis")

In [ ]:
# Train the Logistic Regression model
print("🚀 Training Logistic Regression model...")
print("⏱️  This may take a few minutes depending on dataset size...")

import time
start_time = time.time()

# Fit the model on training data
lr_model = lr.fit(train_data)

training_time = time.time() - start_time

print(f"✅ Model training completed!")
print(f"   • Training time: {training_time:.2f} seconds")
print(f"   • Converged: {lr_model.summary.totalIterations < lr.getMaxIter()}")
print(f"   • Iterations used: {lr_model.summary.totalIterations}/{lr.getMaxIter()}")

# Display training summary
print(f"\n📊 Training Summary:")
print(f"   • Objective history length: {len(lr_model.summary.objectiveHistory)}")
print(f"   • Final objective value: {lr_model.summary.objectiveHistory[-1]:.6f}")

# Get coefficient summary
print(f"   • Model coefficients: {len(lr_model.coefficients)} features")
print(f"   • Intercept: {lr_model.intercept:.6f}")

print(f"\n🎯 Model successfully trained on fraud detection features from ClickHouse!")

## 5. Model Evaluation & Performance Analysis

Evaluate the trained model on test data using fraud detection specific metrics.

In [ ]:
# Make predictions on test data
print("🔮 Making predictions on test dataset...")

# Generate predictions
predictions = lr_model.transform(test_data)

# Cache predictions for multiple evaluations
predictions.cache()

print("✅ Predictions generated")
print(f"   • Test records: {predictions.count():,}")

# Show sample predictions
print(f"\n📋 Sample Predictions:")
predictions.select("label", "prediction", "probability").show(5, truncate=False)

# Quick prediction summary
pred_summary = predictions.groupBy("prediction").count().collect()
print(f"\n📊 Prediction Summary:")
for row in pred_summary:
    pred_label = "Predicted Fraud" if row['prediction'] == 1.0 else "Predicted Legitimate"
    count = row['count']
    percentage = (count / predictions.count()) * 100
    print(f"   • {pred_label}: {count:,} ({percentage:.1f}%)")

In [ ]:
# Comprehensive model evaluation for fraud detection
print("📊 Evaluating model performance...")

# Binary Classification Evaluator for AUC-ROC
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# AUC-ROC Score
auc_score = binary_evaluator.evaluate(predictions)
print(f"🎯 AUC-ROC Score: {auc_score:.4f}")

# AUC-PR Score  
binary_evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
auc_pr_score = binary_evaluator_pr.evaluate(predictions)
print(f"🎯 AUC-PR Score: {auc_pr_score:.4f}")

# Multiclass evaluator for additional metrics
multiclass_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

# Accuracy
accuracy = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "accuracy"})
print(f"🎯 Accuracy: {accuracy:.4f}")

# Precision and Recall for fraud class (class 1)
fraud_precision = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "precisionByLabel", multiclass_evaluator.metricLabel: 1.0})
fraud_recall = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "recallByLabel", multiclass_evaluator.metricLabel: 1.0})
fraud_f1 = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "fMeasureByLabel", multiclass_evaluator.metricLabel: 1.0})

print(f"\n🚨 Fraud Detection Performance:")
print(f"   • Fraud Precision: {fraud_precision:.4f} (of predicted frauds, how many are actual frauds)")
print(f"   • Fraud Recall: {fraud_recall:.4f} (of actual frauds, how many are detected)")  
print(f"   • Fraud F1-Score: {fraud_f1:.4f} (harmonic mean of precision and recall)")

# Performance interpretation
print(f"\n📈 Performance Interpretation:")
if auc_score >= 0.9:
    auc_rating = "Excellent"
elif auc_score >= 0.8:
    auc_rating = "Good" 
elif auc_score >= 0.7:
    auc_rating = "Fair"
else:
    auc_rating = "Needs Improvement"

print(f"   • AUC-ROC ({auc_score:.3f}): {auc_rating}")
print(f"   • Model can distinguish fraud from legitimate transactions")
print(f"   • Fraud detection rate: {fraud_recall:.1%}")
print(f"   • False positive rate: {1-fraud_precision:.1%} (of fraud predictions)")

print(f"\n✅ Model evaluation completed with {auc_rating.lower()} performance for fraud detection")

In [ ]:
# Confusion Matrix and detailed performance analysis
print("📊 Creating confusion matrix for detailed analysis...")

# Create confusion matrix using Spark SQL
predictions.createOrReplaceTempView("predictions_table")

confusion_matrix = spark.sql("""
    SELECT 
        label as actual,
        prediction as predicted,
        COUNT(*) as count
    FROM predictions_table 
    GROUP BY label, prediction
    ORDER BY label, prediction
""").collect()

print(f"\n📋 Confusion Matrix:")
print(f"                    Predicted")
print(f"                Legit    Fraud")
print(f"Actual  Legit    {confusion_matrix[0]['count']:>6}   {confusion_matrix[1]['count']:>6}")
print(f"        Fraud    {confusion_matrix[2]['count']:>6}   {confusion_matrix[3]['count']:>6}")

# Calculate detailed metrics
tn = confusion_matrix[0]['count']  # True Negatives
fp = confusion_matrix[1]['count']  # False Positives  
fn = confusion_matrix[2]['count']  # False Negatives
tp = confusion_matrix[3]['count']  # True Positives

# Calculate business-relevant metrics
false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)
true_positive_rate = tp / (tp + fn)  # Same as recall
true_negative_rate = tn / (tn + fp)  # Specificity

print(f"\n💼 Business Impact Metrics:")
print(f"   • True Positives (Frauds Caught): {tp:,}")
print(f"   • False Negatives (Frauds Missed): {fn:,}")
print(f"   • False Positives (False Alarms): {fp:,}")
print(f"   • True Negatives (Correct Legit): {tn:,}")

print(f"\n📈 Key Rates:")
print(f"   • Fraud Detection Rate: {true_positive_rate:.1%} (caught {tp} of {tp+fn} frauds)")
print(f"   • False Positive Rate: {false_positive_rate:.1%} (false alarms on legit txns)")
print(f"   • False Negative Rate: {false_negative_rate:.1%} (missed frauds)")
print(f"   • Specificity: {true_negative_rate:.1%} (correctly identified legit txns)")

# Cost-benefit analysis (illustrative)
avg_fraud_amount = 50000  # PKR (example)
investigation_cost = 500   # PKR per investigation

fraud_prevented = tp * avg_fraud_amount
investigation_costs = (tp + fp) * investigation_cost
net_benefit = fraud_prevented - investigation_costs

print(f"\n💰 Estimated Financial Impact (Illustrative):")
print(f"   • Fraud Prevented: PKR {fraud_prevented:,}")
print(f"   • Investigation Costs: PKR {investigation_costs:,}")
print(f"   • Net Benefit: PKR {net_benefit:,}")

print(f"\n✅ Detailed performance analysis completed")

## 6. Model Persistence & Deployment Preparation

Save the trained model and prepare for production deployment.

In [ ]:
# Save the trained model and preprocessing pipeline
print("💾 Saving trained model and pipeline...")

import os
from datetime import datetime

# Create model directory
model_base_path = "/root/research-dir/dev/jazzcash-fraud-detection/models"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"{model_base_path}/fraud_logistic_regression_{timestamp}"

# Ensure directory exists
os.makedirs(model_path, exist_ok=True)

# Save the complete trained model
lr_model_path = f"{model_path}/logistic_regression_model"
lr_model.write().overwrite().save(lr_model_path)

# Save the feature scaler
scaler_path = f"{model_path}/feature_scaler"
scaler_model.write().overwrite().save(scaler_path)

# Save feature assembler
assembler_path = f"{model_path}/feature_assembler"
assembler.write().overwrite().save(assembler_path)

print(f"✅ Model components saved:")
print(f"   • Logistic Regression Model: {lr_model_path}")
print(f"   • Feature Scaler: {scaler_path}")
print(f"   • Feature Assembler: {assembler_path}")

# Save model metadata
metadata = {
    "model_type": "LogisticRegression",
    "training_date": timestamp,
    "features_count": len(modeling_features),
    "training_records": train_count,
    "test_records": test_count,
    "auc_score": auc_score,
    "accuracy": accuracy,
    "fraud_precision": fraud_precision,
    "fraud_recall": fraud_recall,
    "selected_features": modeling_features,
    "source_table": TABLE_NAME,
    "clickhouse_database": CLICKHOUSE_CONFIG['database']
}

# Save metadata as JSON
import json
metadata_path = f"{model_path}/model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"   • Model Metadata: {metadata_path}")

print(f"\n📦 Model Package Created: {model_path}")
print(f"🚀 Ready for production deployment!")

# Display deployment readiness checklist
print(f"\n✅ Deployment Readiness Checklist:")
print(f"   ✓ Model trained and validated")
print(f"   ✓ Performance metrics documented")
print(f"   ✓ Model artifacts saved")
print(f"   ✓ Feature pipeline preserved")
print(f"   ✓ ClickHouse connection configured")
print(f"   ✓ Metadata and lineage documented")

print(f"\n🎯 Next Steps for Production:")
print(f"   1. Deploy model to production Spark cluster")
print(f"   2. Create real-time scoring API")
print(f"   3. Set up monitoring and alerting")
print(f"   4. Implement A/B testing framework")
print(f"   5. Schedule model retraining pipeline")

In [ ]:
# Summary and final information
print("=" * 80)
print("🎉 FRAUD DETECTION MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)

print(f"\n📊 FINAL MODEL SUMMARY:")
print(f"   • Algorithm: Logistic Regression (Spark MLlib)")
print(f"   • Data Source: ClickHouse {TABLE_NAME}")
print(f"   • Features: {len(modeling_features)} engineered features")
print(f"   • Training Data: {train_count:,} records")
print(f"   • Test Data: {test_count:,} records")

print(f"\n🎯 PERFORMANCE METRICS:")
print(f"   • AUC-ROC: {auc_score:.4f}")
print(f"   • AUC-PR: {auc_pr_score:.4f}")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Fraud Precision: {fraud_precision:.4f}")
print(f"   • Fraud Recall: {fraud_recall:.4f}")
print(f"   • Fraud F1-Score: {fraud_f1:.4f}")

print(f"\n💼 BUSINESS IMPACT:")
print(f"   • Fraud Detection Rate: {true_positive_rate:.1%}")
print(f"   • False Positive Rate: {false_positive_rate:.1%}")
print(f"   • Estimated Frauds Caught: {tp:,}")
print(f"   • Estimated False Alarms: {fp:,}")

print(f"\n🔧 TECHNICAL DETAILS:")
print(f"   • Feature Engineering: Based on fraud profiling analysis")
print(f"   • Data Pipeline: ClickHouse → Spark → MLlib")
print(f"   • Model Training: {training_time:.1f} seconds")
print(f"   • Convergence: {lr_model.summary.totalIterations} iterations")

print(f"\n📦 ARTIFACTS CREATED:")
print(f"   • Trained Model: {lr_model_path}")
print(f"   • Feature Pipeline: {scaler_path}")
print(f"   • Model Metadata: {metadata_path}")

print(f"\n🚀 PRODUCTION READY:")
print(f"   ✓ Model can be loaded for real-time scoring")
print(f"   ✓ Feature pipeline preserved for consistency")  
print(f"   ✓ ClickHouse integration established")
print(f"   ✓ Performance benchmarks documented")

print(f"\n💡 RECOMMENDATIONS:")
print(f"   • Deploy with fraud_recall threshold optimization")
print(f"   • Monitor model drift with weekly retraining")
print(f"   • Implement A/B testing vs current fraud system")
print(f"   • Set up real-time feature computation pipeline")

print("=" * 80)
print("✅ Fraud detection model ready for production deployment!")
print("📞 Contact MLOps team for production deployment assistance")
print("=" * 80)

# Clean up cached data
train_data.unpersist()
test_data.unpersist()
predictions.unpersist()

print("\n🧹 Cache cleaned up - Spark session ready for reuse")